In [ ]:
import keras
import mltoolkit
from datasets import load_dataset

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

In [2]:
import sys
import os
import importlib
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../src'))
import src.models.amp_freq_resnet18
import src.models.resnet18
importlib.reload(src.models.resnet18)
importlib.reload(src.models.amp_freq_resnet18)

<module 'src.models.amp_freq_resnet18' from 'c:\\Users\\hz\\Documents\\ML\\resnet18_implementation\\src\\models\\amp_freq_resnet18.py'>

In [27]:
from src.models.amp_freq_resnet18 import amp_freq_resnet18_keras

## help pls

In [43]:
from mltoolkit import labels_to_ohe, to_tensor_list

In [53]:
transform_train = transforms.Compose([
    transforms.RandomCrop(64, padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225)),
])

def train_transform(batch):
    batch["image"] = [transform_train(img.convert("RGB")) for img in batch["image"]]
    return batch

def test_transform(batch):
    batch["image"] = [transform_test(img.convert("RGB")) for img in batch["image"]]
    return batch

# def collate_fn(batch):
#     imgs   = torch.stack([b["image"] for b in batch])
#     labels = torch.tensor([b["label"] for b in batch])
#     return imgs, labels

def collate_fn(batch):
    imgs   = torch.stack([b["image"] for b in batch])  # (B, C, H, W)
    imgs   = imgs.permute(0, 2, 3, 1)                  # (B, H, W, C)
    ohe_labels, _ = labels_to_ohe(torch.tensor([b["label"] for b in batch]), num_classes=NUM_CLASSES, return_ohe=True) 
    return imgs, ohe_labels


def get_dataloaders(batch_size=128):
    dataset = load_dataset("zh-plus/tiny-imagenet", cache_dir="../data/raw")
    dataset["train"].set_transform(train_transform)
    dataset["valid"].set_transform(test_transform)

    trainloader = torch.utils.data.DataLoader(
        dataset["train"], batch_size=batch_size, shuffle=True, pin_memory=True,
        collate_fn=collate_fn)

    testloader = torch.utils.data.DataLoader(
        dataset["valid"], batch_size=batch_size, shuffle=False, pin_memory=True,
        collate_fn=collate_fn)

    return trainloader, testloader

In [29]:

def train_imagnet(model, epochs=60, lr=0.001, weight_decay=5e-4, batch_size=128):
    """
    Modern minimal training loop for CIFAR-100
    Good starting point for most conv / ViT / ResNet / ConvNeXt / EfficientNetV2 models
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    trainloader, testloader = get_dataloaders(batch_size)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    print(f"Training on {device} — {len(trainloader.dataset)} train images, {len(testloader.dataset)} test")
    print(f"Batch size = {batch_size}, epochs = {epochs}, base LR = {lr}\n")

    best_acc = 0.0

    for epoch in range(epochs):
        model.train()
        correct = total = 0
        running_loss = 0.0

        for i, (images, labels) in enumerate(trainloader):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            # stats
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            if (i + 1) % 100 == 0:
                print(f"[{epoch+1:2d}/{epochs}] [{i+1:5d}/{len(trainloader)}]  "
                      f"loss: {running_loss/100:.3f}  |  acc: {100.*correct/total:.2f}%")
                running_loss = correct = total = 0.0

        # ──── Evaluation ───────────────────────────────────────
        model.eval()
        correct = total = 0

        with torch.no_grad():
            for images, labels in testloader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                outputs = model(images)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        acc = 100. * correct / total
        print(f"Epoch {epoch+1:2d} ─ Test acc: {acc:.2f}%")

        if acc > best_acc:
            best_acc = acc
            print(f"↑ New best: {best_acc:.2f}%")

        scheduler.step()

    print(f"\nFinished. Best test accuracy: {best_acc:.2f}%")
    return model, best_acc

## too late

In [49]:
EPOCHS = 30
BATCH_SIZE = 128
LR = 1e-3
NUM_CLASSES = 200

In [36]:
model = amp_freq_resnet18_keras((64, 64, 3), num_classes=NUM_CLASSES, amp_factors=[8,6,4,2], thresh_quantiles=[0.2, 0.4, 0.6, 0.8])

In [54]:
train_loader, val_loader = get_dataloaders(BATCH_SIZE)

In [ ]:


mltoolkit.create_run_dir('run_1_tiny_imagenet')b


datasets.dataset_dict.DatasetDict

In [55]:
model.compile(
    optimizer=keras.optimizers.AdamW(),
    loss=keras.losses.CategoricalCrossentropy()
)

In [56]:
model.fit(train_loader, validation_data=val_loader, epochs=3)

Epoch 1/3
118/782 ━━━━━━━━━━━━━━━━━━━━ 1:02 94ms/step - loss: 7.2295

KeyboardInterrupt: 